In [12]:
# importing the libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [13]:
# Reading the dataset
dataset = pd.read_csv("Social_Network_Ads.csv")
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [14]:
# Change categorical data to nominal
dataset = pd.get_dummies(dataset, drop_first=True)
dataset

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,True
1,15810944,35,20000,0,True
2,15668575,26,43000,0,False
3,15603246,27,57000,0,False
4,15804002,19,76000,0,True
...,...,...,...,...,...
395,15691863,46,41000,1,False
396,15706071,51,23000,1,True
397,15654296,50,20000,1,False
398,15755018,36,33000,0,True


In [15]:
# Split independent and dependent variables
independent = dataset[["Age", "EstimatedSalary", "Gender_Male"]]
dependent = dataset[["Purchased"]]

In [16]:
# Split into training set and test set
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(independent, dependent, test_size=1/3, random_state=0)

In [17]:
# Import StandardScaler to standardize the features
from sklearn.preprocessing import StandardScaler

# Create a StandardScaler object
sc = StandardScaler()

# Fit the scaler using the training data and transform the training data
# Each feature will be scaled to have mean = 0 and standard deviation = 1
x_train = sc.fit_transform(x_train)

# Use the same scaler fitted on training data to transform the test data
# We use only transform() here to avoid data leakage from the test set
x_test = sc.transform(x_test)

In [18]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

param_grid = {
    'kernel': [
        'linear',
        'rbf',
        'poly',
        'sigmoid'
    ],
    'gamma': [
        'auto',
        'scale'
    ],
    'C': [
        10, 100, 1000, 2000, 3000
    ]
}

grid = GridSearchCV(
    SVC(probability=True), 
    param_grid, 
    refit=True, 
    verbose=3,
    cv=5,
    n_jobs=-1,
    scoring='f1_weighted'
)

grid.fit(x_train, y_train)

Fitting 5 folds for each of 40 candidates, totalling 200 fits


C:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


,estimator,SVC(probability=True)
,param_grid,"{'C': [10, 100, ...], 'gamma': ['auto', 'scale'], 'kernel': ['linear', 'rbf', ...]}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,5
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,C,100


In [20]:
# print best parameter after tuning
print(grid.best_params_)

re = grid.cv_results_
print(re)

grid_predictions = grid.predict(x_test)

# print confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_predictions)
print("Confusion Matrix:\n", cm) 

# print classification report
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)
print("CFM Report:\n ", clf_report)

{'C': 100, 'gamma': 'auto', 'kernel': 'rbf'}
{'mean_fit_time': array([0.00721874, 0.00561824, 0.0087409 , 0.00464377, 0.0052    ,
       0.00492716, 0.00659003, 0.0042417 , 0.01643257, 0.00847764,
       0.02750726, 0.00536761, 0.01997004, 0.00940876, 0.0252852 ,
       0.00500836, 0.10521808, 0.01983814, 0.15900211, 0.00631785,
       0.1206923 , 0.0234055 , 0.17819996, 0.00725217, 0.25287466,
       0.03837314, 0.35048881, 0.00619402, 0.21757512, 0.03650036,
       0.37700605, 0.00577817, 0.29470568, 0.05034218, 0.59892302,
       0.00636592, 0.33495393, 0.05925841, 0.64041271, 0.00828552]), 'std_fit_time': array([0.00163564, 0.00164893, 0.00168756, 0.0014797 , 0.00039269,
       0.00137087, 0.00074636, 0.00107306, 0.00263656, 0.00197743,
       0.01088963, 0.00066995, 0.0043765 , 0.00114088, 0.00779833,
       0.00084957, 0.02485234, 0.00106845, 0.03969185, 0.00177873,
       0.031706  , 0.00458954, 0.07996045, 0.00200818, 0.06624807,
       0.00830263, 0.12025144, 0.00243154, 0.049

In [21]:
# Import f1_score to calculate the F1-score
from sklearn.metrics import f1_score

# Calculate the weighted F1-score
# y_test          -> Actual values
# grid_predictions -> Predictions made by the best GridSearchCV model
# average='weighted' -> Calculate F1 for each class and weight it based on class size
f1_weighted = f1_score(
    y_test,
    grid_predictions,
    average='weighted'
)

# Display the best parameters found by GridSearchCV
# and the F1-score obtained on the test data
print(
    "The f1_weighted value for best parameter {}:".format(grid.best_params_),
    f1_weighted
)


# Import roc_auc_score to calculate ROC-AUC
from sklearn.metrics import roc_auc_score

# Calculate ROC-AUC score
# ROC-AUC evaluates how well your model can distinguish between the two classes.
# grid.predict_proba(x_test) returns probabilities for each class
# [:, 1] selects the probability of class 1
roc_auc_score(
    y_test,
    # Get probability of class 1
    grid.predict_proba(x_test)[:, 1]
)

The f1_weighted value for best parameter {'C': 100, 'gamma': 'auto', 'kernel': 'rbf'}: 0.9100355779243318


0.9539015606242497